# Lambdas

This module covers Python's `lambda` expression: when to reach for one, when a regular function reads better, and where lambdas naturally appear in tm1py work. Readers are assumed to be comfortable with Python functions and with at least passing familiarity with TM1 cubes and the tm1py library; the focus here is on the cases that actually come up when shaping cube data, sorting elements, and feeding helpers like `sorted`, `min`, `max`, `filter`, and `pandas.DataFrame.apply`.

A lambda is a small anonymous function written as a single expression. The TM1 cellset returned by `execute_view_values` as a `dict[tuple[str, ...], float]` keyed by element names is the running example throughout the module; sorting, filtering, and reshaping that dict is where most of the lambdas in real tm1py code live.

The topics below are arranged linearly for review. Structural grouping (sections, chapters) can be applied later.

---

## Topic list

1. Lambdas in Python
2. Anatomy of a lambda
3. Lambda vs `def`
4. Lambdas as sort keys
5. `min`, `max`, and `sorted` on cube data
6. Lambdas with `filter` and `map`
7. Closures: what a lambda captures
8. Late binding inside loops
9. Lambdas in pandas transforms
10. Type hints and the limits of lambda
11. Real world design principles
12. Common mistakes

---

## 1. Lambdas in Python

A `lambda` expression produces a function object without binding it to a name. The function has no statements, only one expression, whose value is returned. Every other function feature carries over: positional and keyword parameters, default values, closures over enclosing variables, and use as a first class value. What is left out is everything that requires a statement: assignment, `if`/`else` blocks, `for` loops, `try`/`except`, and a `return` statement. A lambda is a function compressed to its expression.

The form exists because Python takes "functions are values" seriously enough to give a literal syntax for them. When a function is small enough to fit on one line and is used in exactly one place, naming it through `def` is overhead the lambda removes. Most lambdas in working Python code are one of three things: a sort key, a callback handed to a library, or a small predicate fed to `filter`, `groupby`, or a pandas method.

## 2. Anatomy of a lambda

A lambda has the shape `lambda parameters: expression`. The result is a function object identical to one produced by `def`, except that `__name__` is the placeholder string `"<lambda>"`.

In [ ]:
double = lambda x: x * 2
double(5)            # 10
double.__name__      # '<lambda>'

# multiple parameters, default value, *args, **kwargs all work
weighted = lambda value, factor=1.0: value * factor
combine = lambda *parts, sep=" | ": sep.join(parts)

weighted(120_000.0, 1.1)               # 132000.0
combine("2026", "Europe", "Phones")    # '2026 | Europe | Phones'

The body is a single expression. There is no place in a lambda for `return` (the expression is the return value) and no place for a statement of any kind. Conditional logic comes through the conditional expression `a if cond else b`, not an `if` block:

In [ ]:
clamp = lambda value: 0.0 if value < 0 else value
clamp(-50.0)         # 0.0
clamp(120_000.0)     # 120000.0

A lambda is otherwise a plain function. It can be assigned to a name (rarely a good idea, see §3), passed as an argument, returned from another function, or stored in a list.

## 3. Lambda vs `def`

The two forms produce equivalent function objects. The choice is stylistic.

In [ ]:
# def form
def double(x: int) -> int:
    return x * 2

# lambda form
double = lambda x: x * 2

Two reasons argue against assigning a lambda to a name. First, the function's `__name__` stays `"<lambda>"`, which makes tracebacks and debugger output less useful. Second, a `def` admits type hints and a docstring; a lambda admits neither. The PEP 8 style guide says directly: "always use a `def` statement instead of an assignment statement that binds a lambda expression directly to an identifier."

The right place for a lambda is inline, where the function is the argument to another call and is not used anywhere else:

In [ ]:
years: list[str] = ["2026", "2025", "2024", "2027"]
sorted(years, key=lambda year: int(year))
# ['2024', '2025', '2026', '2027']

The lambda exists only as the `key` argument. Naming it would mean reading two lines instead of one, and the name would not be reused.

## 4. Lambdas as sort keys

The most common place for a lambda in tm1py work is the `key` argument of `sorted`, `list.sort`, `min`, and `max`. These functions accept a one argument callable that maps each item to the value to compare on.

A cellset arrives as a dict whose keys are tuples of element names. Sorting cells by value, descending:

In [ ]:
cells: dict[tuple[str, ...], float] = tm1.cells.execute_view_values(
    cube_name="Sales Plan", view_name="Plan Input"
)
# cells: { ("2026", "Jan", "Europe", "Phones", "Plan", "Revenue"): 120_000.0,
#          ("2026", "Feb", "Europe", "Phones", "Plan", "Revenue"): 135_000.0, ... }

top_cells = sorted(cells.items(), key=lambda item: item[1], reverse=True)
# [(("2026", "Mar", "Europe", "Phones", "Plan", "Revenue"), 152_000.0),
#  (("2026", "Feb", "Europe", "Phones", "Plan", "Revenue"), 135_000.0), ...]

Sorting by tuple slots in the key (region, then period) works the same way, with a tuple as the key value:

In [ ]:
by_region_then_period = sorted(
    cells.items(),
    key=lambda item: (item[0][2], item[0][1]),
)

For this very common shape, the standard library offers `operator.itemgetter`, which produces the same callable without a lambda:

In [ ]:
from operator import itemgetter
top_cells = sorted(cells.items(), key=itemgetter(1), reverse=True)

`itemgetter` is faster (it is implemented in C) and reads better when the only operation is index or key lookup. Reach for a lambda when the key needs a small computation that `itemgetter` cannot express:

In [ ]:
# absolute value of variance, ignoring sign
by_size = sorted(variances.items(), key=lambda item: abs(item[1]), reverse=True)

## 5. `min`, `max`, and `sorted` on cube data

`min` and `max` accept the same `key` argument and benefit from lambdas in the same way. The largest revenue cell in a cellset:

In [ ]:
peak_key, peak_value = max(cells.items(), key=lambda item: item[1])
# (("2026", "Mar", "Europe", "Phones", "Plan", "Revenue"), 152_000.0)

The closest cell to a target value, used for picking a benchmark period:

In [ ]:
target = 130_000.0
nearest = min(cells.items(), key=lambda item: abs(item[1] - target))
# (("2026", "Feb", "Europe", "Phones", "Plan", "Revenue"), 135_000.0)

The `key` callable runs once per item, so an expensive lookup inside it is paid once per item, not once per comparison. This is what makes `key=` cheap even for large cellsets where a comparator based approach would be quadratic.

`sorted` returns a new list; `list.sort` sorts in place and returns `None`. Element name lists from tm1py come back as ordinary lists, so either form applies. The in place form is appropriate when the original ordering is not needed:

In [ ]:
periods = list(tm1.elements.get_element_names("Period"))
periods.sort(key=lambda name: ("Total" in name, name))
# pushes any element whose name contains "Total" to the end

The pattern of returning a tuple from the key, where the leading element is a flag, is how stable sorts implement "group A first, then group B, alphabetical inside each group" without writing two passes.

## 6. Lambdas with `filter` and `map`

`filter(fn, iterable)` and `map(fn, iterable)` accept a callable, which in inline use is almost always a lambda. Both return iterators in Python 3, so a `list(...)` wrap is needed to materialize:

In [ ]:
revenue_keys = list(filter(lambda key: key[5] == "Revenue", cells))
revenue_values = list(map(lambda v: v * 1.1, cells.values()))

A list comprehension expresses the same shapes more directly, and is what most modern Python code uses:

In [ ]:
revenue_keys = [key for key in cells if key[5] == "Revenue"]
revenue_values = [v * 1.1 for v in cells.values()]

The comprehension wins because the body of a comprehension is an ordinary expression with full access to the surrounding scope, while `map` and `filter` need a callable, which invites a lambda. `map` is still idiomatic when the callable is already named (`map(str.upper, names)` rather than `[s.upper() for s in names]`); a `map` or `filter` whose callable is a lambda is almost always a comprehension in disguise.

The one place lambdas with `map` and `filter` still earn their keep is when a callable is wanted as a value, not as the inline transformation. A reducer that takes a predicate, a chain of `itertools` calls, a `functools.reduce` accumulator: these all want a function, and a lambda is the lightest way to provide one.

## 7. Closures: what a lambda captures

A lambda defined inside another function can refer to variables from the enclosing scope. The lambda holds a reference to the variable, not a copy of its value, and looks it up when the lambda is called.

In [ ]:
def revenue_filter(target_region: str):
    return lambda key: key[2] == target_region and key[5] == "Revenue"

is_europe_revenue = revenue_filter("Europe")
europe_keys = [key for key in cells if is_europe_revenue(key)]

`target_region` is captured by reference. The lambda sees whatever value the name has when the lambda runs, not when it was defined. For ordinary use this is invisible: the enclosing function returns and `target_region` is fixed for the lifetime of the closure. The case where it bites is creating lambdas inside a loop, covered next.

A closure can also capture data computed once and reuse it across many calls:

In [ ]:
def in_known_region(tm1_service):
    known = set(tm1_service.elements.get_element_names("Region"))
    return lambda key: key[2] in known

The set is built once and captured; every call to the returned lambda reuses it. This is the lightweight version of a small class with one method, and is appropriate when the closure has only one or two captured values.

## 8. Late binding inside loops

Closures capture by reference, which produces a classic surprise when lambdas are built inside a loop:

In [ ]:
predicates = []
for region in ["Europe", "Asia", "Americas"]:
    predicates.append(lambda key: key[2] == region)

probe = ("2026", "Jan", "Europe", "Phones", "Plan", "Revenue")
[p(probe) for p in predicates]
# [False, False, False]

All three lambdas closed over the same `region` variable; by the time they run, the loop has finished and `region` is `"Americas"`, so none of them match a probe whose region is `"Europe"`. The fix is to bind the value at definition time through a default argument, which is evaluated when the lambda is created:

In [ ]:
predicates = []
for region in ["Europe", "Asia", "Americas"]:
    predicates.append(lambda key, region=region: key[2] == region)

[p(probe) for p in predicates]
# [True, False, False]

The `region=region` idiom looks redundant but is the standard Python solution. The default value is evaluated immediately and stored on the function object; the lambda's body still uses the parameter name. The same trick works for any loop variable that needs to be frozen into the closure, and applies equally to lambdas built inside a comprehension.

## 9. Lambdas in pandas transforms

When a cellset is loaded into a pandas DataFrame, lambdas appear as the callable for `Series.apply`, `DataFrame.apply`, `Series.map`, `groupby(...).agg`, and `assign`. These are exactly the places where the transformation is small, one off, and not worth naming.

In [ ]:
import pandas as pd

df = pd.DataFrame(
    [(*key, value) for key, value in cells.items()],
    columns=["Year", "Period", "Region", "Product", "Version", "Measure", "Value"],
)

# bucket cells by size
df["Tier"] = df["Value"].apply(lambda v: "Large" if v >= 100_000 else "Small")

# combine columns with a small computation
df["Slice"] = df.apply(lambda row: f"{row['Region']} / {row['Product']}", axis=1)

Two notes specific to pandas. First, vectorized operations are almost always faster than `.apply` with a lambda; `df["Value"] * 1.1` beats `df["Value"].apply(lambda v: v * 1.1)` by a wide margin because the vector form runs in compiled C, while the lambda form calls back into Python once per row. Reach for a lambda only when the transformation cannot be expressed as a vector operation. Second, `DataFrame.apply(..., axis=1)` is the slowest common pandas idiom; if a per row computation is unavoidable, a list comprehension over `df.itertuples()` is usually faster and reads as well.

`groupby(...).agg` accepts lambdas for custom aggregations:

In [ ]:
spread_by_region = df.groupby("Region")["Value"].agg(lambda values: values.max() - values.min())
# Region
# Americas    32000.0
# Asia        28500.0
# Europe      32000.0

This is a fine use: the callable is short, used once, and the named alternative would only repeat the expression with a name attached.

## 10. Type hints and the limits of lambda

A lambda has no place to write type annotations. The parameters cannot be annotated, the return cannot be annotated, and there is no docstring. Tools that depend on annotations (mypy, pyright, IDE inference) fall back to inferring the types from context, which works for inline use as a `key=` argument but is less reliable as soon as the lambda is stored, returned, or passed across module boundaries.

In [ ]:
# def admits annotations
def by_value(item: tuple[tuple[str, ...], float]) -> float:
    return item[1]

# lambda does not
by_value = lambda item: item[1]    # type checker infers from caller, if at all

For any callable whose signature deserves to be documented, or that is exported, or that is reused, a `def` is the right form. The lambda's value is brevity at the call site; once the function is meaningful enough to be named, brevity is no longer the right tradeoff.

A second limit is statements. A lambda body cannot assign, cannot loop, cannot use `try`/`except`, and cannot raise an exception except through a function call that itself raises. The conditional expression `a if cond else b` covers a surprising amount of ground, but a lambda that is reaching for chained `if`/`else` ladders is a function in disguise.

## 11. Real world design principles

**Prefer a comprehension to `map` or `filter` with a lambda.** When the body is anything more than a single named function call, the comprehension reads more directly and gives the same iterator (or list, set, dict) back.

**Reach for `operator.itemgetter` and `operator.attrgetter` first.** Most lambdas in `key=` use are doing index lookup or attribute access, both of which the `operator` module expresses without a lambda and slightly faster.

**Inline only.** A lambda earns its place when it is the argument to another call and is not reused. Once it is bound to a name, returned from a function, or used twice, a `def` is the better form.

**Bind loop variables explicitly.** The `name=name` default argument idiom is the Python answer to late binding. Use it whenever a lambda is built inside a loop, even when the surrounding loop "should" be safe; safety here depends on whether the lambda is called before or after the loop ends, which is a fragile invariant to rely on.

**Do not reach for a lambda where a vectorized operation exists.** This applies to numpy, pandas, and any tm1py call that can take a slice of a cellset directly. The lambda is a per element callback; the vectorized form is one operation on the whole array, and the difference on real cellsets is a factor of ten or more.

**Write the named version when reviewers might pause.** A lambda whose body is not obvious at a glance is a function in disguise. Naming it costs three lines of code and saves the reader the parse.

## 12. Common mistakes

**Assigning a lambda to a name.**

In [ ]:
# Wrong
double = lambda x: x * 2

# Correct
def double(x: int) -> int:
    return x * 2

**Using `return` inside a lambda.**

In [ ]:
# Wrong: SyntaxError, return is a statement
key_fn = lambda item: return item[1]

# Correct: the expression is the return value
key_fn = lambda item: item[1]

**Forgetting late binding when building lambdas in a loop.**

In [ ]:
# Wrong: every predicate sees the last region
predicates = [lambda key: key[2] == r for r in ["Europe", "Asia", "Americas"]]

# Correct: bind r at definition time
predicates = [lambda key, r=r: key[2] == r for r in ["Europe", "Asia", "Americas"]]

**Reaching for a lambda when `operator` already covers the case.**

In [ ]:
# Wrong
top_cells = sorted(cells.items(), key=lambda item: item[1], reverse=True)

# Correct
from operator import itemgetter
top_cells = sorted(cells.items(), key=itemgetter(1), reverse=True)

**Using `map` or `filter` with a lambda where a comprehension is clearer.**

In [ ]:
# Wrong
revenue_keys = list(filter(lambda key: key[5] == "Revenue", cells))

# Correct
revenue_keys = [key for key in cells if key[5] == "Revenue"]

**Calling `.apply` with a lambda where pandas already vectorizes.**

In [ ]:
# Wrong
df["Value"] = df["Value"].apply(lambda v: v * 1.1)

# Correct
df["Value"] = df["Value"] * 1.1

**Building a multi line conditional inside a lambda.**

In [ ]:
# Wrong: the lambda is a function in disguise
classify = lambda v: ("Large" if v >= 100_000
                      else "Medium" if v >= 50_000
                      else "Small" if v >= 0
                      else "Negative")

# Correct
def classify(value: float) -> str:
    if value < 0:
        return "Negative"
    if value < 50_000:
        return "Small"
    if value < 100_000:
        return "Medium"
    return "Large"